# Training: one job, before we orchestrate it

The pipeline's TrainingStep is just a training job — run it once directly to see the pieces before `Pipeline` wraps them.

In [ ]:
import io

import boto3
import pandas as pd
from feast import FeatureStore
from wine_quality_classification.feature_contract import (
    FEATURE_COLUMNS,
    LABEL_COLUMN,
)

s3 = boto3.client(
    "s3", endpoint_url="http://wine-quality-classification-moto:5000"
)
BUCKET = "wine-quality-classification"
try:
    s3.create_bucket(Bucket=BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    pass

store = FeatureStore(repo_path="feature_repo")
entities = pd.read_parquet("data/processed/wine_labels.parquet")
training = store.get_historical_features(
    entity_df=entities,
    features=[f"wine_assay_profile:{c}" for c in FEATURE_COLUMNS]
    + [f"wine_cultivar_labels:{LABEL_COLUMN}"],
).to_df()

buf = io.BytesIO()
training[FEATURE_COLUMNS + [LABEL_COLUMN]].to_csv(buf, index=False)
s3.put_object(Bucket=BUCKET, Key="input/train/train.csv", Body=buf.getvalue())
len(training)

## The job

Image, role, hyperparameters — the same call shape the pipeline step reuses.

In [ ]:
import json

from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession

session = LocalSession()
estimator = Estimator(
    image_uri="wine-quality-classification:train",
    role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
    instance_count=1,
    instance_type="local",
    output_path=f"s3://{BUCKET}/output",
    sagemaker_session=session,
    hyperparameters={"C": 1.0, "max_iter": 1000},
)
estimator.set_entry_point(
    source_dir="src/wine_quality_classification", entry_point="train.py"
)
estimator.fit({"train": f"s3://{BUCKET}/input/train"})
json.loads(
    s3.get_object(
        Bucket=BUCKET,
        Key=f"output/{estimator.latest_training_job.name}/output/metrics.json",
    )["Body"].read()
)